# Pivots and Joins

Run the next cell once to install the packages this notebook uses. Then run the `# datascience` and `# polars` import cells. If an import still fails, restart the kernel and run from the top.


In [ ]:
# setup
# Pins match data8-textbook-polars/requirements.txt.
%pip install -q \
    numpy==2.5.4 \
    pandas==3.1.0 \
    pyarrow==26.0.0 \
    scipy==1.18.1 \
    matplotlib==3.11.2 \
    plotly==7.1.0 \
    datascience==0.18.1 \
    polars==2.1.0 \
    hvplot==0.12.2


In [1]:
# datascience
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
import warnings
warnings.simplefilter(action='ignore',category=np.exceptions.VisibleDeprecationWarning)


In [2]:
# polars
import polars as pl


## Rows from lists

In [3]:
# datascience
Table().with_columns('Numbers', [1, 2, 3])


Numbers
1
2
3


In [4]:
# polars
pl.DataFrame([
    pl.Series('Numbers', [1, 2, 3])
])


Numbers
i64
1
2
3


In [5]:
# datascience
drinks = Table(['Drink', 'Cafe', 'Price'])
drinks


Drink,Cafe,Price


In [6]:
# polars
drinks_pl = pl.DataFrame(schema={'Drink': pl.String, 'Cafe': pl.String, 'Price': pl.Float64})
drinks_pl


Drink,Cafe,Price
str,str,f64


In [7]:
# datascience
drinks = drinks.with_rows([
    ['Milk Tea', 'Asha', 5.5],
    ['Espresso', 'Strada',  1.75],
    ['Latte',    'Strada',  3.25],
    ['Espresso', "FSM",   2]
])
drinks


Drink,Cafe,Price
Milk Tea,Asha,5.5
Espresso,Strada,1.75
Latte,Strada,3.25
Espresso,FSM,2


In [8]:
# polars
drinks_pl = pl.concat([
    drinks_pl,
    pl.DataFrame([
        pl.Series('Drink', ['Milk Tea', 'Espresso', 'Latte', 'Espresso']),
        pl.Series('Cafe', ['Asha', 'Strada', 'Strada', 'FSM']),
        pl.Series('Price', [5.5, 1.75, 3.25, 2])
    ])
])
drinks_pl


Drink,Cafe,Price
str,str,f64
"""Milk Tea""","""Asha""",5.5
"""Espresso""","""Strada""",1.75
"""Latte""","""Strada""",3.25
"""Espresso""","""FSM""",2.0


## Cross-classification: grouping by two columns

Do right-handed people tend to sleep on their left side and left-handed people sleep on their right?

In [9]:
# datascience
survey = Table.read_table('welcome_survey_fa26.csv')
survey.show(3)


Extroversion,Text Recipients,Sleep Hours,Handedness,Sleep Position,Piercings
7,5,6,Right-handed,On your right side,0
7,10,7,Ambidextrous,On your right side,2
3,7,9,Left-handed,On your left side,0


In [10]:
# polars
survey_pl = pl.read_csv('welcome_survey_fa26.csv')
survey_pl.show(3)


Extroversion,Text Recipients,Sleep Hours,Handedness,Sleep Position,Piercings
i64,i64,f64,str,str,i64
7,5,6.0,"""Right-handed""","""On your right side""",0
7,10,7.0,"""Ambidextrous""","""On your right side""",2
3,7,9.0,"""Left-handed""","""On your left side""",0


In [11]:
# datascience
survey.group(['Handedness', 'Sleep Position']).show()


Handedness,Sleep Position,count
Ambidextrous,On your back,3
Ambidextrous,On your left side,1
Ambidextrous,On your right side,5
I can write with both hands and feet,On your back,1
Left-handed,On your back,20
Left-handed,On your left side,33
Left-handed,On your right side,29
Left-handed,On your stomach,13
Right-handed,On your back,309
Right-handed,On your left side,267


In [12]:
# polars
survey_pl.group_by(['Handedness', 'Sleep Position']).len(name='count').sort(['Handedness', 'Sleep Position']).show(None)


Handedness,Sleep Position,count
str,str,u32
"""Ambidextrous""","""On your back""",3
"""Ambidextrous""","""On your left side""",1
"""Ambidextrous""","""On your right side""",5
"""I can write with both hands an…","""On your back""",1
"""Left-handed""","""On your back""",20
"""Left-handed""","""On your left side""",33
"""Left-handed""","""On your right side""",29
"""Left-handed""","""On your stomach""",13
"""Right-handed""","""On your back""",309


In [13]:
# datascience
survey.group(['Handedness', 'Sleep Position'], np.average).show()


Handedness,Sleep Position,Extroversion average,Text Recipients average,Sleep Hours average,Piercings average
Ambidextrous,On your back,6.66667,10.6667,7.33333,0
Ambidextrous,On your left side,3,1,6,2
Ambidextrous,On your right side,5.4,4.8,6.7,1
I can write with both hands and feet,On your back,1,12,8,0
Left-handed,On your back,5.45,10.6,6.75,0.7
Left-handed,On your left side,6.06061,7.78788,7.06061,0.878788
Left-handed,On your right side,6.10345,7.44828,7.18966,1.48276
Left-handed,On your stomach,5.38462,8.76923,7.38462,2.30769
Right-handed,On your back,5.72168,6.60841,7.11327,0.977346
Right-handed,On your left side,5.58427,6.8764,6.89963,1.68165


In [14]:
# polars
survey_pl.group_by(['Handedness', 'Sleep Position']).mean().sort(['Handedness', 'Sleep Position']).show(None)


Handedness,Sleep Position,Extroversion,Text Recipients,Sleep Hours,Piercings
str,str,f64,f64,f64,f64
"""Ambidextrous""","""On your back""",6.666667,10.666667,7.333333,0.0
"""Ambidextrous""","""On your left side""",3.0,1.0,6.0,2.0
"""Ambidextrous""","""On your right side""",5.4,4.8,6.7,1.0
"""I can write with both hands an…","""On your back""",1.0,12.0,8.0,0.0
"""Left-handed""","""On your back""",5.45,10.6,6.75,0.7
"""Left-handed""","""On your left side""",6.060606,7.787879,7.060606,0.878788
"""Left-handed""","""On your right side""",6.103448,7.448276,7.189655,1.482759
"""Left-handed""","""On your stomach""",5.384615,8.769231,7.384615,2.307692
"""Right-handed""","""On your back""",5.721683,6.608414,7.113269,0.977346


## Pivot Tables

In [15]:
# datascience
survey.group('Handedness')


Handedness,count
Ambidextrous,9
I can write with both hands and feet,1
Left-handed,95
Right-handed,1017


In [16]:
# polars
survey_pl.group_by('Handedness').len(name='count').sort('Handedness')


Handedness,count
str,u32
"""Ambidextrous""",9
"""I can write with both hands an…",1
"""Left-handed""",95
"""Right-handed""",1017


In [17]:
# datascience
survey.pivot('Sleep Position', 'Handedness')


Handedness,On your back,On your left side,On your right side,On your stomach
Ambidextrous,3,1,5,0
I can write with both hands and feet,1,0,0,0
Left-handed,20,33,29,13
Right-handed,309,267,334,107


In [18]:
# polars
sleep_by_hand_pl = survey_pl.pivot('Sleep Position', index='Handedness', values='Sleep Hours', aggregate_function='len')
value_cols = sorted(col for col in sleep_by_hand_pl.columns if col != 'Handedness')
sleep_by_hand_pl.select('Handedness', *value_cols).sort('Handedness').fill_null(0)


Handedness,On your back,On your left side,On your right side,On your stomach
str,u32,u32,u32,u32
"""Ambidextrous""",3,1,5,0
"""I can write with both hands an…",1,0,0,0
"""Left-handed""",20,33,29,13
"""Right-handed""",309,267,334,107


**DISCUSS**: Why do the group and pivot tables have different numbers of entries?

In [19]:
# datascience
survey.pivot('Sleep Position', 
             'Handedness', 
             values='Sleep Hours', 
             collect=np.average)


Handedness,On your back,On your left side,On your right side,On your stomach
Ambidextrous,7.33333,6,6.7,0
I can write with both hands and feet,8,0,0,0
Left-handed,6.75,7.06061,7.18966,7.38462
Right-handed,7.11327,6.89963,7.07635,7.06075


In [20]:
# polars
sleep_hours_by_hand_pl = survey_pl.pivot('Sleep Position', index='Handedness', values='Sleep Hours', aggregate_function='mean')
value_cols = sorted(col for col in sleep_hours_by_hand_pl.columns if col != 'Handedness')
sleep_hours_by_hand_pl.select('Handedness', *value_cols).sort('Handedness').fill_null(0)


Handedness,On your back,On your left side,On your right side,On your stomach
str,f64,f64,f64,f64
"""Ambidextrous""",7.333333,6.0,6.7,0.0
"""I can write with both hands an…",8.0,0.0,0.0,0.0
"""Left-handed""",6.75,7.060606,7.189655,7.384615
"""Right-handed""",7.113269,6.899625,7.076347,7.060748


**DISCUSS**: In the above pivot table, what does one number mean?

## Discussion Questions ##

In [21]:
# datascience
# From the CORGIS Dataset Project
# By Austin Cory Bart acbart@vt.edu
# Version 2.0.0, created 3/22/2016
# https://corgis-edu.github.io/corgis/csv/skyscrapers/

sky = Table.read_table('skyscrapers.csv')
sky = (sky.with_column('age', 2022 - sky.column('completed'))
          .drop('completed'))
sky.show(3)


name,material,city,height,age
One World Trade Center,mixed/composite,New York City,541.3,8
Willis Tower,steel,Chicago,442.14,48
432 Park Avenue,concrete,New York City,425.5,7


In [22]:
# polars
# From the CORGIS Dataset Project
# By Austin Cory Bart acbart@vt.edu
# Version 2.0.0, created 3/22/2016
# https://corgis-edu.github.io/corgis/csv/skyscrapers/

sky_pl = pl.read_csv('skyscrapers.csv')
sky_pl = (sky_pl.with_columns((2022 - pl.col('completed')).alias('age'))
          .drop('completed'))
sky_pl.show(3)


name,material,city,height,age
str,str,str,f64,i64
"""One World Trade Center""","""mixed/composite""","""New York City""",541.299988,8
"""Willis Tower""","""steel""","""Chicago""",442.140015,48
"""432 Park Avenue""","""concrete""","""New York City""",425.5,7


### 1. For each city, what’s the height of the tallest building for each material?

In [23]:
# YOUR CODE HERE



### 2. For each city, what’s the height difference between the tallest steel building and the tallest concrete building?

In [24]:
# YOUR CODE HERE












---

#### **WARNING: ANSWERS BELOW**

In [25]:
# datascience
sky.select('material', 'city', 'height').group(['city', 'material'], collect=max)


city,material,height max
Atlanta,concrete,264.25
Atlanta,mixed/composite,311.8
Atlanta,steel,169.47
Austin,concrete,208.15
Austin,steel,93.6
Baltimore,concrete,161.24
Baltimore,steel,155.15
Boston,concrete,121.92
Boston,mixed/composite,139
Boston,steel,240.79


In [26]:
# polars
sky_pl.select('material', 'city', 'height').group_by(['city', 'material']).max().sort(['city', 'material'])


city,material,height
str,str,f64
"""Atlanta""","""concrete""",264.25
"""Atlanta""","""mixed/composite""",311.799988
"""Atlanta""","""steel""",169.470001
"""Austin""","""concrete""",208.149994
"""Austin""","""steel""",93.599998
…,…,…
"""Seattle""","""steel""",235.309998
"""St. Louis""","""concrete""",100.660004
"""St. Louis""","""mixed/composite""",180.75


In [27]:
# datascience
tallest = sky.pivot('material', 'city', values='height', collect=max)
tallest.show()


city,concrete,mixed/composite,steel
Atlanta,264.25,311.8,169.47
Austin,208.15,0,93.6
Baltimore,161.24,0,155.15
Boston,121.92,139,240.79
Charlotte,265.48,239.7,179.23
Chicago,423.22,306.94,442.14
Cincinnati,125,202.69,175
Cleveland,125,288.65,215.8
Columbus,79.25,0,169.3
Dallas,176.48,280.72,270.06


In [28]:
# polars
tallest_pl = (
    sky_pl.pivot('material', index='city', values='height', aggregate_function='max')
    .sort('city')
    .fill_null(0)
)
tallest_pl.show(None)


city,mixed/composite,steel,concrete
str,f64,f64,f64
"""Atlanta""",311.799988,169.470001,264.25
"""Austin""",0.0,93.599998,208.149994
"""Baltimore""",0.0,155.149994,161.240005
"""Boston""",139.0,240.789993,121.919998
"""Charlotte""",239.699997,179.229996,265.480011
"""Chicago""",306.940002,442.140015,423.220001
"""Cincinnati""",202.690002,175.0,125.0
"""Cleveland""",288.649994,215.800003,125.0
"""Columbus""",0.0,169.300003,79.25


In [29]:
# datascience
diffs = Table().with_columns(
    'city', tallest.column('city'),
    'difference', abs(tallest.column('steel') - tallest.column('concrete'))
)
diffs


city,difference
Atlanta,94.78
Austin,114.55
Baltimore,6.09001
Boston,118.87
Charlotte,86.25
Chicago,18.92
Cincinnati,50
Cleveland,90.8
Columbus,90.05
Dallas,93.58


In [30]:
# polars
diffs_pl = tallest_pl.select(
    'city',
    (pl.col('steel') - pl.col('concrete')).abs().alias('difference')
)
diffs_pl


city,difference
str,f64
"""Atlanta""",94.779999
"""Austin""",114.549995
"""Baltimore""",6.090012
"""Boston""",118.869995
"""Charlotte""",86.250015
…,…
"""San Diego""",0.909988
"""San Francisco""",40.839996
"""Seattle""",96.619995


In [31]:
# datascience
diffs.sort('difference', descending=True)


city,difference
Sunny Isles Beach,196
Las Vegas,185.62
Miami Beach,170.39
Pittsburgh,167.04
Los Angeles,164.59
Philadelphia,130.15
Boston,118.87
Austin,114.55
Seattle,96.62
Atlanta,94.78


In [32]:
# polars
diffs_pl.sort('difference', descending=True)


city,difference
str,f64
"""Sunny Isles Beach""",196.0
"""Las Vegas""",185.619995
"""Miami Beach""",170.389999
"""Pittsburgh""",167.039993
"""Los Angeles""",164.590012
…,…
"""Salt Lake City""",13.720001
"""Portland""",11.270004
"""Baltimore""",6.090012


### 3. Generate a table of the names of the oldest buildings for each material for each city.

In [33]:
# datascience
# Hint: You can use sort to find the name of the oldest building in the dataset
sky.sort('age', descending=True).column('name').item(0)

# YOUR CODE HERE


'Statue of Liberty'

In [34]:
# polars
# Hint: You can use sort to find the name of the oldest building in the dataset
sky_pl.sort('age', descending=True).get_column('name').item(0)

# YOUR CODE HERE


'Statue of Liberty'

---

#### **WARNING: ANSWERS BELOW**

In [35]:
# datascience
def first(s):
    "Return the first element in an array."
    return s.item(0)

(sky
 .sort('age', descending=True)
 .pivot('material', 'city', 'name', first)
)#.where('city', 'San Francisco')


city,concrete,mixed/composite,steel
Atlanta,Westin Peachtree Plaza,One Atlantic Center,FlatironCity
Austin,One American Center,,University of Texas Tower
Baltimore,Charles Towers North Apartments,,Emerson Tower
Boston,Harbor Towers I,Ellison Building,Marriott's Custom House
Charlotte,Bank of America Corporate Center,Hearst Tower,Midtown Plaza
Chicago,The Powhatan,American Furniture Mart,The Rookery
Cincinnati,Kroger Building,Great American Tower at Queen City Square,PNC Tower
Cleveland,National City Center,55 Public Square,Huntington Bank Building
Columbus,Key Bank Building,,Leveque Tower
Dallas,Reunion Tower,Bank of America Plaza,Three AT&T Plaza


In [36]:
# polars
def first_pl(s):
    "Return the first element in an array."
    return s.item(0)

(sky_pl
 .sort('age', descending=True)
 .pivot('material', index='city', values='name', aggregate_function='first')
 .sort('city')
)#.filter(pl.col('city') == 'San Francisco')


city,steel,concrete,mixed/composite
str,str,str,str
"""Atlanta""","""FlatironCity""","""Westin Peachtree Plaza""","""One Atlantic Center"""
"""Austin""","""University of Texas Tower""","""One American Center""",null
"""Baltimore""","""Emerson Tower""","""Charles Towers North Apartment…",null
"""Boston""","""Marriott's Custom House""","""Harbor Towers I""","""Ellison Building"""
"""Charlotte""","""Midtown Plaza""","""Bank of America Corporate Cent…","""Hearst Tower"""
…,…,…,…
"""San Diego""","""Executive Complex""","""Meridian Condominiums""","""Electra"""
"""San Francisco""","""Ferry Building""","""Coit Tower""","""Transamerica Pyramid"""
"""Seattle""","""Smith Tower""","""Continental Place""","""Unigard Financial Center"""


## Joins ##

In [37]:
# datascience
drinks


Drink,Cafe,Price
Milk Tea,Asha,5.5
Espresso,Strada,1.75
Latte,Strada,3.25
Espresso,FSM,2


In [38]:
# polars
drinks_pl


Drink,Cafe,Price
str,str,f64
"""Milk Tea""","""Asha""",5.5
"""Espresso""","""Strada""",1.75
"""Latte""","""Strada""",3.25
"""Espresso""","""FSM""",2.0


In [39]:
# datascience
discounts = Table().with_columns(
    'Coupon % off', make_array(10, 25, 5),
    'Location', make_array('Asha', 'Strada', 'Asha')
)
discounts


Coupon % off,Location
10,Asha
25,Strada
5,Asha


In [40]:
# polars
discounts_pl = pl.DataFrame([
    pl.Series('Coupon % off', [10, 25, 5]),
    pl.Series('Location', ['Asha', 'Strada', 'Asha'])
])
discounts_pl


Coupon % off,Location
i64,str
10,"""Asha"""
25,"""Strada"""
5,"""Asha"""


In [41]:
# datascience
combined = drinks.join('Cafe', discounts, 'Location')
combined


Cafe,Drink,Price,Coupon % off
Asha,Milk Tea,5.5,10
Asha,Milk Tea,5.5,5
Strada,Espresso,1.75,25
Strada,Latte,3.25,25


In [42]:
# polars
combined_pl = drinks_pl.join(discounts_pl, left_on='Cafe', right_on='Location').sort('Cafe')
combined_pl


Drink,Cafe,Price,Coupon % off
str,str,f64,i64
"""Milk Tea""","""Asha""",5.5,10
"""Milk Tea""","""Asha""",5.5,5
"""Espresso""","""Strada""",1.75,25
"""Latte""","""Strada""",3.25,25


In [43]:
# datascience
discounted_frac = 1 - combined.column('Coupon % off') / 100
combined.with_column(
    'Discounted Price', 
    combined.column('Price') * discounted_frac
)


Cafe,Drink,Price,Coupon % off,Discounted Price
Asha,Milk Tea,5.5,10,4.95
Asha,Milk Tea,5.5,5,5.225
Strada,Espresso,1.75,25,1.3125
Strada,Latte,3.25,25,2.4375


In [44]:
# polars
discounted_frac_pl = 1 - combined_pl.get_column('Coupon % off') / 100
combined_pl.with_columns((combined_pl.get_column('Price') * discounted_frac_pl).alias('Discounted Price'))


Drink,Cafe,Price,Coupon % off,Discounted Price
str,str,f64,i64,f64
"""Milk Tea""","""Asha""",5.5,10,4.95
"""Milk Tea""","""Asha""",5.5,5,5.225
"""Espresso""","""Strada""",1.75,25,1.3125
"""Latte""","""Strada""",3.25,25,2.4375


In [45]:
# datascience
drinks.join('Cafe', drinks, 'Cafe')


Cafe,Drink,Price,Drink_2,Price_2
Asha,Milk Tea,5.5,Milk Tea,5.5
FSM,Espresso,2,Espresso,2
Strada,Espresso,1.75,Espresso,1.75
Strada,Espresso,1.75,Latte,3.25
Strada,Latte,3.25,Espresso,1.75
Strada,Latte,3.25,Latte,3.25


In [46]:
# polars
drinks_pl.join(drinks_pl, on='Cafe')


Drink,Cafe,Price,Drink_right,Price_right
str,str,f64,str,f64
"""Milk Tea""","""Asha""",5.5,"""Milk Tea""",5.5
"""Espresso""","""Strada""",1.75,"""Espresso""",1.75
"""Latte""","""Strada""",3.25,"""Espresso""",1.75
"""Espresso""","""Strada""",1.75,"""Latte""",3.25
"""Latte""","""Strada""",3.25,"""Latte""",3.25
"""Espresso""","""FSM""",2.0,"""Espresso""",2.0
